# Responsive CSS Extraction and Optimization
This notebook audits `public/assets/css/style.css`, extracts responsive `@media` rules into `public/assets/css/responsive.css`, and checks for duplicate selectors while preserving the current design.

In [ ]:
from pathlib import Path
import re
root = Path('public/assets/css')
style_path = root / 'style.css'
responsive_path = root / 'responsive.css'
print('style.css exists:', style_path.exists())
print('responsive.css exists:', responsive_path.exists())
style_text = style_path.read_text(encoding='utf-8')
print('style.css length:', len(style_text), 'chars')
print('Total @media occurrences in style.css:', len(re.findall(r'@media', style_text)))
print('Total @media occurrences in responsive.css:', len(re.findall(r'@media', responsive_path.read_text(encoding='utf-8'))))

In [ ]:
# Identify top-level @media blocks in style.css
media_spans = []
pattern = re.compile(r'@media\b')
text = style_text
n = len(text)
idx = 0
for m in pattern.finditer(text):
    start = m.start()
    if start < idx:
        continue
    brace = 0
    pos = start
    while pos < n:
        ch = text[pos]
        if ch == '{':
            brace += 1
        elif ch == '}':
            brace -= 1
            if brace == 0:
                end = pos + 1
                break
        pos += 1
    else:
        raise SystemExit(f'Unmatched brace at {start}')
    media_spans.append((start, end))
    idx = end
print('Found media blocks:', len(media_spans))
print('First 5 media blocks starting positions:', [s for s, e in media_spans[:5]])

In [ ]:
# Extract and move @media blocks to responsive.css
existing_responsive = responsive_path.read_text(encoding='utf-8')
if not existing_responsive.endswith('\n'):
    existing_responsive += '\n'
existing_responsive += '\n/* Extracted responsive rules from style.css */\n'
new_style = []
last = 0
for start, end in media_spans:
    new_style.append(text[last:start])
    existing_responsive += text[start:end] + '\n\n'
    last = end
new_style.append(text[last:])
new_style_text = ''.join(new_style)
print('New style.css length:', len(new_style_text))
print('Extracted', len(media_spans), 'blocks')
# style_path.write_text(new_style_text, encoding='utf-8')
# responsive_path.write_text(existing_responsive, encoding='utf-8')
print('Prepared texts for rewrite.')

In [ ]:
# Detect duplicate selectors across style.css and responsive.css
selector_pattern = re.compile(r'([^\{]+)\{[^\}]*\}')
def extract_selectors(css_text):
    selectors = []
    for match in selector_pattern.finditer(css_text):
        selector = match.group(1).strip()
        if selector.startswith('@media'):
            continue
        selectors.append(selector)
    return selectors
style_selectors = extract_selectors(style_text)
res_selectors = extract_selectors(existing_responsive)
print('Selectors in style.css:', len(style_selectors))
print('Selectors in responsive.css after extraction placeholder:', len(res_selectors))
print('Common selector count:', len(set(style_selectors) & set(res_selectors)))

In [ ]:
print('style.css @media count after extraction placeholder:', len(re.findall(r'@media', new_style_text)))
print('responsive.css @media count after extraction placeholder:', len(re.findall(r'@media', existing_responsive)))